# Zero-Trust Privacy-Aware Agent Runtime
## Multimodal Memory Gate — Presentation Demo
**Authors:** Yilin Pan & Tyler Hudgins | CS 466 | April 2026

---

### Innovation: Stage-Aware, Multimodal Privacy Control

Existing tools (e.g., Microsoft Presidio, text-only NER) treat privacy as **one-time text preprocessing**.
This system addresses a fundamentally harder problem: *privacy across modalities, across pipeline stages,
before content reaches durable memory.*

| Capability | Text-Only Tools | **This System** |
|---|---|---|
| PII in text (NER) | ✅ | ✅ |
| PII in images (visual objects) | ❌ | ✅ OwlViT zero-shot |
| PII in OCR text extracted from images | ❌ | ✅ Ephemeral OCR |
| PII across PDF pages | ❌ | ✅ Page-by-page |
| Stage-aware policy (ALLOW / MASK / ABSTRACT) | ❌ | ✅ |
| Utility preservation via generative inpainting | ❌ | ✅ Stable Diffusion |
| Zero cloud inference — all models local | ❌ | ✅ |

### Formal Model: π(E, s) → a
Privacy control is a function of **detected entities E** and **system stage s**:
- `E` — sensitive findings (text spans, image bounding boxes, OCR regions)
- `s ∈ {SENSING, MEMORY_WRITE, RETRIEVAL, OUTPUT}`
- `a ∈ {ALLOW, MASK, ABSTRACT, BLOCK}`

A key insight from Kim et al. (2026) *Attack and Defense Landscape of Agentic AI*:
> Agent **memory** is a first-class attack surface — retrieval-stage leakage can
> expose sensitive content that was supposedly "stored safely."

Our system inserts gates **before vectorization**, controlling what enters durable memory.


---
## Phase 0: Setup & Configuration

In [ ]:
import os, sys, time, yaml
import matplotlib.pyplot as plt

_nb_dir = os.path.abspath("")
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

print("Standard library imports OK")

In [ ]:
from src.privacy_pipeline import (
    initialize_models, setup_output_dir, get_models,
    detect_privacy_risks, privacy_gate_and_embed, redact_text,
    load_images_from_path, visualize_detection_result,
    visualize_comparison, visualize_utility,
)
from src.evaluation.baseline_comparison import compare_all, compute_utility_score
from src.config.settings import OPENAI_API_KEY

with open("config/test_inputs.yaml") as f:
    cfg = yaml.safe_load(f)

TEST_TEXTS     = cfg["test_texts"]
TEST_IMAGES    = cfg["test_images"]
TEST_PDFS      = cfg["test_pdfs"]
VISUAL_QUERIES = [cfg["sensitive_visual_queries"]]

OUTPUTS = setup_output_dir()
print(f"  Test texts   : {len(TEST_TEXTS)}")
print(f"  Test images  : {len(TEST_IMAGES)}")
print(f"  Test PDFs    : {len(TEST_PDFS)}")
print(f"  Outputs →    {OUTPUTS}")

---
## Phase 0: Setup & Configuration

All heavy pipeline logic lives in `src/privacy_pipeline.py`.
This notebook imports and runs it — no function definitions here.

| Model | Role |
|---|---|
| `openai/privacy-filter` | NER-based text PII token classification |
| `EasyOCR` | Locate text bounding boxes in images (ephemeral — strings never stored) |
| `google/owlvit-base-patch32` | Zero-shot visual object detection |
| `runwayml/stable-diffusion-inpainting` | Generative inpainting for MASK policy (768×768 canvas) |
| `openai/clip-vit-base-patch32` | Safe CLIP embedding after gate |

In [ ]:
initialize_models()
models = get_models()
print(f"\nDevice: {models['device']} | All models ready.")

---
## Demo 1: Text PII Detection

The text gate runs the `openai/privacy-filter` NER model over input text and replaces PII spans
with typed tags such as `[ACCOUNT_NUMBER]`, `[PRIVATE_ADDRESS]`, `[NAME]`.

This prevents sensitive tokens from ever reaching the CLIP embedding or being stored in the vector DB.

In [ ]:
print("=" * 72)
print("  DEMO 1 — Text PII Detection (batch)")
print("=" * 72)

for i, text in enumerate(TEST_TEXTS, 1):
    t0 = time.perf_counter()
    redacted, has_pii = redact_text(text)
    elapsed = time.perf_counter() - t0
    status = "✅ PII found" if has_pii else "✔  Clean"
    print(f"\n  [{i}] {status}  ({elapsed:.2f}s)")
    print(f"  Original : {text}")
    print(f"  Redacted : {redacted}")

---
## Demo 2: Multimodal Input — Text + Image

When input contains both text and an image, the system:
1. Redacts PII in the text (NER)
2. Detects sensitive regions in the image (OwlViT + ephemeral OCR)
3. Applies stage-aware policy: **ALLOW** / **MASK** / **ABSTRACT**
4. Generates safe CLIP embeddings for the (now-clean) memory entry

In [ ]:
demo2_text      = TEST_TEXTS[0]
demo2_image_cfg = TEST_IMAGES[0]
demo2_path      = demo2_image_cfg["path"]

print("=" * 72)
print(f"  DEMO 2 — Text + Image  ({demo2_image_cfg['description']})")
print("=" * 72)

t_start = time.perf_counter()
results  = detect_privacy_risks(demo2_text, demo2_path)
r        = results[0]
policy, final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
    image=r["image"], redacted_text=r["redacted_text"], sensitive_boxes=r["sensitive_boxes"],
)
t_total = time.perf_counter() - t_start

print(f"\n  Policy          : {policy}")
print(f"  Original text   : {demo2_text}")
print(f"  Redacted text   : {safe_text}")
print(f"  System note     : {summary}")
print(f"  Sensitive boxes : {len(r['sensitive_boxes'])} region(s)")
print(f"  Total time      : {t_total:.2f}s")

out_dir = OUTPUTS / "demo2_text_image"
out_dir.mkdir(exist_ok=True)
fig = visualize_detection_result(
    r, policy=policy, final_image=final_img,
    title="Demo 2 — Multimodal Privacy Gate",
    save_path=out_dir / "scenario_figure.png",
)
plt.show()

print("\n  Safe CLIP embeddings (stored in memory):")
if embeddings["image_embeds"] is not None:
    print(f"    Image : {tuple(embeddings['image_embeds'].shape)}")
if embeddings["text_embeds"] is not None:
    print(f"    Text  : {tuple(embeddings['text_embeds'].shape)}")

---
## Demo 3: Baseline Comparison — Why Multimodal?

A central claim of this work is that **text-only PII detection is not sufficient for multimodal inputs.**

This section runs four detection modes on the same input and compares what each catches:

| Mode | Text PII | Visual PII | OCR PII | Expected Verdict |
|---|---|---|---|---|
| A — Text-Only | ✅ | ❌ missed | ❌ missed | INCOMPLETE |
| B — Visual-Only (OwlViT) | ❌ missed | ✅ | ❌ missed | INCOMPLETE |
| C — Visual + OCR | ❌ missed | ✅ | ✅ | INCOMPLETE |
| **D — Full Pipeline (Ours)** | **✅** | **✅** | **✅** | **COMPLETE** |

> Reference: Zhao (WebPII, ICLR 2026) shows text-extraction baselines achieve **0.357 mAP@50**
> vs visual detection at **0.753 mAP@50** on UI screenshots — combining both modalities is substantially stronger.

In [ ]:
print("=" * 72)
print("  DEMO 3 — Baseline Comparison: Email Screenshot")
print("=" * 72)

email_cfg   = next(c for c in TEST_IMAGES if "email" in c["path"])
email_image = load_images_from_path(email_cfg["path"])[0]
demo3_text  = "My name is Sarah Chen, please save this screenshot."

results = compare_all(
    demo3_text, email_image,
    models["pii_filter"], models["owl_processor"], models["owl_model"],
    models["device"], models["reader"],
    queries=VISUAL_QUERIES,
)

print(f"  Input text : {demo3_text}")
print(f"  {'Mode':<30}  {'Text':^6}  {'Visual':^8}  {'Boxes':^5}  {'Time':^8}")
print("  " + "─" * 64)
for r in results:
    t_sym = "✅" if r["text_pii_found"] else "–"
    v_sym = "✅" if r["visual_pii_found"] else "–"
    print(f"  {r['mode_name']:<30}  {t_sym:^6}  {v_sym:^8}  {r['n_boxes']:^5}  {r['elapsed_s']:^8.2f}")

out_dir = OUTPUTS / "comparison"
out_dir.mkdir(exist_ok=True)
fig = visualize_comparison(
    results, email_image,
    title=f"Baseline Comparison — {email_cfg['description']}",
    save_path=out_dir / "email_comparison_figure.png",
)
plt.show()

In [ ]:
print("=" * 72)
print("  DEMO 3 — Baseline Comparison: Face Photo")
print("=" * 72)

face_cfg   = next(c for c in TEST_IMAGES if "face" in c["path"])
face_image = load_images_from_path(face_cfg["path"])[0]

results = compare_all(
    None, face_image,
    models["pii_filter"], models["owl_processor"], models["owl_model"],
    models["device"], models["reader"],
    queries=VISUAL_QUERIES,
)

print(f"  (No text input — text-only mode catches nothing)")
print(f"  {'Mode':<30}  {'Text':^6}  {'Visual':^8}  {'Boxes':^5}  {'Time':^8}")
print("  " + "─" * 64)
for r in results:
    t_sym = "✅" if r["text_pii_found"] else "–"
    v_sym = "✅" if r["visual_pii_found"] else "–"
    print(f"  {r['mode_name']:<30}  {t_sym:^6}  {v_sym:^8}  {r['n_boxes']:^5}  {r['elapsed_s']:^8.2f}")

fig = visualize_comparison(
    results, face_image,
    title=f"Baseline Comparison — {face_cfg['description']}",
    save_path=out_dir / "face_comparison_figure.png",
)
plt.show()

---
## Demo 4: Evaluation Metrics

We measure two key dimensions:

**Privacy (how much leakage was prevented)**
- `n_boxes` — number of sensitive regions caught per image
- `policy` — action taken (ALLOW / MASK / ABSTRACT)
- `text_pii` — whether text PII was caught

**Utility (how much useful content was preserved)**
- `utility_score` — CLIP cosine similarity between the original image embedding
  and the post-gate embedding (1.0 = identical semantics, 0 = fully different)
  - ALLOW: 1.0 (image unchanged)
  - MASK: high (≈ 0.85–0.98 — inpainting preserves scene context)
  - ABSTRACT: N/A (image suppressed; text summary stored instead)

**Processing time** — per-image latency for the full pipeline

> **Why CLIP similarity?**  
> If we simply black-boxed or suppressed everything, utility would collapse.
> Generative inpainting (MASK policy) preserves the semantic context of the image
> so downstream retrieval and reasoning can still use it meaningfully.
> This is the core privacy–utility tradeoff argument.

In [ ]:
print("=" * 78)
print("  DEMO 4 — Per-Image Evaluation Metrics")
print("=" * 78)
print(f"  {'File':<30} {'Policy':<10} {'Boxes':^6} {'Text PII':^10} {'Utility':^10} {'Time (s)':^10}")
print("  " + "─" * 76)

_image_gate_cache = []

for img_cfg in TEST_IMAGES:
    path  = img_cfg["path"]
    fname = os.path.basename(path)
    images = load_images_from_path(path)
    if not images:
        continue
    img = images[0]

    t0 = time.perf_counter()
    results  = detect_privacy_risks(image_path=path)
    r        = results[0]
    boxes    = r["sensitive_boxes"]
    policy, final_img, summary, embeddings, _ = privacy_gate_and_embed(
        image=img, redacted_text="", sensitive_boxes=boxes,
    )
    elapsed = time.perf_counter() - t0

    if policy == "ALLOW":
        utility = 1.0
    elif policy == "MASK" and final_img is not None:
        utility = compute_utility_score(
            img, final_img,
            models["clip_processor"], models["clip_model"], models["device"],
        )
    else:
        utility = float("nan")

    _image_gate_cache.append({
        "img_cfg": img_cfg, "img": img, "boxes": boxes,
        "policy": policy, "final_img": final_img,
        "utility": utility, "elapsed": elapsed,
    })

    utility_str = f"{utility:.3f}" if utility == utility else "N/A (suppressed)"
    print(f"  {fname:<30} {policy:<10} {len(boxes):^6} {'–':^10} {utility_str:^10} {elapsed:^10.2f}")

print("  " + "─" * 76)
print("\n  Utility = CLIP cosine similarity (original vs. post-gate). ALLOW=1.0, ABSTRACT=N/A.")

In [ ]:
print("Utility Visualization: original vs. masked image (MASK policy)\n")

mask_demos = [
    (e["img"], e["final_img"], e["utility"],
     os.path.basename(e["img_cfg"]["path"]), e["boxes"])
    for e in _image_gate_cache
    if e["policy"] == "MASK" and e["final_img"] is not None
]

if mask_demos:
    util_dir = OUTPUTS / "utility"
    util_dir.mkdir(exist_ok=True)

    for orig, masked, score, fname, boxes in mask_demos:
        base = os.path.splitext(fname)[0]
        fig = visualize_utility(
            orig, masked, boxes, score, fname=fname,
            save_path=util_dir / f"{base}_utility_figure.png",
        )
        plt.show()
else:
    print("No MASK-policy images in current test set.")

---
## Demo 5: Privacy-Safe Agent Response

After the privacy gate, the redacted content is passed to an LLM agent.
The key argument: **the agent receives only safe, PII-free input** — yet can still produce
a natural, context-sensitive, and useful response.

Compare:
- **Baseline** (no gate): raw text with PII → LLM → response may reproduce PII
- **Gated** (ours): redacted text → LLM → helpful response, no PII exposure

The privacy gate does not break agent utility — it preserves it.

In [ ]:
def generate_safe_agent_response(redacted_text: str, api_key: str = "") -> str:
    """Generate a natural response using only the post-gate (PII-free) content.

    The LLM never sees the original text — only the redacted version.
    """
    if not api_key:
        return (
            "[No API key — placeholder response]\n\n"
            "I've noted your information. The sensitive details have been safely protected "
            "and will not be stored in our memory system. "
            "I can still help you with the parts of your request that don't require that information."
        )
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful privacy-aware assistant. "
                    "The user's input has had sensitive personal information replaced with [TYPE] tags "
                    "by an upstream privacy gate. "
                    "Respond naturally and helpfully based on the available context. "
                    "Do NOT ask for, guess, or attempt to recover the redacted information. "
                    "Do NOT mention that PII was redacted unless it directly affects your ability to help. "
                    "Focus on what you CAN help with."
                ),
            },
            {"role": "user", "content": redacted_text},
        ],
        max_tokens=200,
    )
    return completion.choices[0].message.content or ""


print("Agent response function defined.")

In [ ]:
print("=" * 72)
print("  DEMO 5 — Privacy-Safe Agent Response")
print("=" * 72)

for i, original_text in enumerate(TEST_TEXTS[:3], 1):
    print(f"\n  ── Test {i} {'─' * 58}")
    redacted, has_pii = redact_text(original_text)
    response = generate_safe_agent_response(redacted, api_key=OPENAI_API_KEY)

    print(f"\n  ORIGINAL  : {original_text}")
    print(f"  REDACTED  : {redacted}")
    print(f"  PII FOUND : {'YES — gate applied' if has_pii else 'No PII detected'}")
    print(f"\n  AGENT RESPONSE (receives only redacted text):")
    print(f"  {'─' * 60}")
    for line in response.split("\n"):
        print(f"  {line}")

---
## Demo 6: Image + PDF Gallery

Full multimodal pipeline across all test images and PDFs.

In [ ]:
all_test_paths = [(c["path"], c["description"]) for c in TEST_IMAGES] + \
                 [(c["path"], c["description"]) for c in TEST_PDFS]

print("=" * 72)
print("  DEMO 6 — Image & PDF Gallery")
print("=" * 72)

gallery_dir = OUTPUTS / "demo6_gallery"
gallery_dir.mkdir(exist_ok=True)

for file_path, description in all_test_paths:
    fname = os.path.basename(file_path)
    base  = os.path.splitext(fname)[0]
    print(f"\n  Processing: {fname} — {description}")

    results = detect_privacy_risks(user_text=None, image_path=file_path)

    for r in results:
        page_idx  = r["page_index"]
        page_lbl  = f"_p{page_idx + 1}" if page_idx is not None else ""
        page_disp = f" (page {page_idx + 1})" if page_idx is not None else ""

        policy, final_img, summary, embeddings, _ = privacy_gate_and_embed(
            image=r["image"], sensitive_boxes=r["sensitive_boxes"],
        )
        print(f"  Policy: {policy} | Boxes: {len(r['sensitive_boxes'])} | {summary}")

        if r["image"] is None:
            continue

        fig = visualize_detection_result(
            r, policy=policy, final_image=final_img,
            title=f"{fname}{page_disp}",
            save_path=gallery_dir / f"{base}{page_lbl}_figure.png",
        )
        plt.show()

---
## Summary: Architecture Contributions

| Contribution | What it does | Why it matters |
|---|---|---|
| **Stage-aware policy π(E, s) → a** | Different privacy actions at different pipeline stages | Retrieval can still leak even if storage was controlled |
| **Ephemeral OCR** | OCR text strings are immediately deleted after use | Prevents text embedded in images from entering vector DB |
| **OwlViT zero-shot detection** | Detects sensitive visual objects without fine-tuning | Catches faces, passports, credit cards that text NER cannot |
| **Generative inpainting (MASK)** | Stable Diffusion replaces sensitive regions | Preserves scene semantics — CLIP utility ≈ 0.85–0.98 vs 0.0 for suppression |
| **ABSTRACT policy** | High-density sensitive images replaced with text summary | Prevents vector embedding of highly sensitive visual content |
| **Zero cloud inference** | All privacy decisions made locally | No PII ever leaves the machine — true zero-trust privacy gate |

### Privacy–Utility Tradeoff
The system targets the region where privacy is high AND utility is preserved:
- Text-only tools: low visual privacy, moderate text utility
- ABSTRACT-only: high privacy, low utility (destroys semantic content)
- **This system**: high privacy across modalities + utility-preserving MASK policy for partial coverage cases

### Threat Model Coverage
Following Kim et al. (2026), the system addresses attacks via all agent memory components:
- **Input gate (SENSING)**: detects PII before it enters the pipeline
- **Memory write gate**: controls what reaches the vector DB and object store
- **Output gate**: safe agent responses never contain original PII